In [ ]:
from google.colab import drive                            #This can be neglected if you're running from local machine
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install opencv-python-headless scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 24.0 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torch.nn.functional as F

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels + skip_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True)
        )
    def forward(self, x, skip=None):
        x = self.upsample(x)
        if skip is not None:
            if x.shape[2:] != skip.shape[2:]:
                x = nn.functional.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
            x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class TransUNetHybrid(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights=None)
        self.encoder1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.encoder2 = nn.Sequential(resnet.maxpool, resnet.layer1)
        self.encoder3 = resnet.layer2
        self.encoder4 = resnet.layer3
        self.encoder5 = resnet.layer4
        self.transformer_bridge = nn.Sequential(nn.Conv2d(2048, 512, kernel_size=1), nn.GroupNorm(32, 512), nn.ReLU(inplace=True))
        self.dec1, self.dec2 = DecoderBlock(512, 1024, 256), DecoderBlock(256, 512, 128)
        self.dec3, self.dec4 = DecoderBlock(128, 256, 64), DecoderBlock(64, 0, 32)
        self.final_head = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        c1, c2, c3, c4, c5 = self.encoder1(x), self.encoder2(self.encoder1(x)), self.encoder3(self.encoder2(self.encoder1(x))), self.encoder4(self.encoder3(self.encoder2(self.encoder1(x)))), self.encoder5(self.encoder4(self.encoder3(self.encoder2(self.encoder1(x)))))
        x = self.dec4(self.dec3(self.dec2(self.dec1(self.transformer_bridge(c5), c4), c3), c2))
        return nn.functional.interpolate(self.final_head(x), size=(512, 512), mode='bilinear', align_corners=True)

# --- 4. LOSS FUNCTIONS ---
class HybridLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
    def forward(self, pred, target):
        p = torch.sigmoid(pred)
        dice = 1.0 - (2. * (p * target).sum(dim=(2, 3)) + 1e-5) / (p.sum(dim=(2, 3)) + target.sum(dim=(2, 3)) + 1e-5)
        return self.bce(pred, target) + dice.mean()

print("Success: Technical utilities and explicit TransUNetHybrid architecture compiled in memory.")

Success: Technical utilities and explicit TransUNetHybrid architecture compiled in memory.


In [ ]:
import os
import cv2
import numpy as np
import torch
from sklearn.metrics import accuracy_score, jaccard_score, cohen_kappa_score, precision_score, recall_score, f1_score

device = torch.device('cpu')

# Directory Pathway Mappings (Configured for your Google Drive Layout)
DATASET_ROOT = "/content/drive/MyDrive/GLOFEAGLES CHALLENGE/GLOFEagles Challenge Validation Dataset"                      #Change local path address of the validation dataset,if needed
IMAGE_ROOT = os.path.join(DATASET_ROOT, "images")
LABEL_ROOT = os.path.join(DATASET_ROOT, "labels")
OUTPUT_ROOT = "/content/drive/MyDrive/GLOFEAGLES CHALLENGE/Validation_Predictions_V2"                                 #Change Output folder address ,if needed

# Updated to target your specific training runs folder path string
WEIGHTS_PATH = "/content/drive/MyDrive/GLOFEAGLES CHALLENGE/runs/train/fine_tuned/weights/best_finetuned.pt"             #Address of the saved model .pt file

# Instantiate Model Architecture
model = TransUNetHybrid()
if os.path.exists(WEIGHTS_PATH):
    model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
    model.to(device)
    model.eval()
    print(f"Success: Pre-trained weights loaded from path: {WEIGHTS_PATH}")
else:
    raise FileNotFoundError(f"Verification Error: Target file missing at specified location: {WEIGHTS_PATH}")

def calculate_all_metrics(pred, target):
    """
    Computes all 6 evaluation metrics required for dense semantic segmentation evaluation.
    """
    pred_flat = (pred > 0).astype(np.uint8).flatten()
    target_flat = (target > 0).astype(np.uint8).flatten()

    acc = accuracy_score(target_flat, pred_flat)
    iou = jaccard_score(target_flat, pred_flat, zero_division=1)
    kappa = cohen_kappa_score(target_flat, pred_flat)
    precision = precision_score(target_flat, pred_flat, zero_division=1)
    recall = recall_score(target_flat, pred_flat, zero_division=1)
    f1 = f1_score(target_flat, pred_flat, zero_division=1)

    return acc, iou, kappa, precision, recall, f1

if not os.path.exists(IMAGE_ROOT):
    raise FileNotFoundError(f"Verification Failure: Ensure dataset shortcuts exist at: {IMAGE_ROOT}")

sub_categories = [f for f in os.listdir(IMAGE_ROOT) if os.path.isdir(os.path.join(IMAGE_ROOT, f))]
validation_summary_log = {}

print(f"Detected Target Sub-categories: {sub_categories}")

for category in sub_categories:
    print(f"\nEvaluating Category: {category}")
    cat_img_dir = os.path.join(IMAGE_ROOT, category)
    cat_lbl_dir = os.path.join(LABEL_ROOT, category)
    cat_out_dir = os.path.join(OUTPUT_ROOT, category)
    os.makedirs(cat_out_dir, exist_ok=True)

    category_metrics_accumulator = []
    target_files = [f for f in os.listdir(cat_img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    for filename in target_files:
        img_path = os.path.join(cat_img_dir, filename)
        raw_scene = cv2.imread(img_path)
        if raw_scene is None:
            continue
        orig_h, orig_w = raw_scene.shape[:2]

        # Preprocessing Matrix Stream Transformation
        rgb_scene = cv2.cvtColor(raw_scene, cv2.COLOR_BGR2RGB)
        resized_scene = cv2.resize(rgb_scene, (512, 512))
        normalized_scene = resized_scene.astype(np.float32) / 255.0
        input_tensor = torch.from_numpy(normalized_scene).permute(2, 0, 1).unsqueeze(0).to(device)

        # CPU Model Inference Execution Pass
        with torch.no_grad():
            output_logits = model(input_tensor)
            probability_mask = torch.sigmoid(output_logits).squeeze().numpy()

        # Post-processing Optimization to Target Matrix Dimensions
        binary_mask = (probability_mask >= 0.5).astype(np.uint8) * 255
        final_mask = cv2.resize(binary_mask, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)

        # Export Predicted Array Mask
        cv2.imwrite(os.path.join(cat_out_dir, filename), final_mask)

        # Metric Extraction Pass
        lbl_path = os.path.join(cat_lbl_dir, filename)
        if os.path.exists(lbl_path):
            ground_truth_mask = cv2.imread(lbl_path, cv2.IMREAD_GRAYSCALE)
            if ground_truth_mask is not None:
                ground_truth_mask = cv2.resize(ground_truth_mask, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
                metrics_tuple = calculate_all_metrics(final_mask, ground_truth_mask)
                category_metrics_accumulator.append(metrics_tuple)

    if category_metrics_accumulator:
        mean_category_scores = np.mean(category_metrics_accumulator, axis=0)
        validation_summary_log[category] = mean_category_scores
        print(f"Category metrics calculation completed for: {category}")

# Global Structured Metrics Matrix Output Logging
print("\n==========================================================================================")
print("                       FINAL COMPREHENSIVE 6-METRIC EVALUATION MATRIX                     ")
print("==========================================================================================")
print(f"{'Category Profile':18} | {'Accuracy':8} | {'Mean IoU':8} | {'Kappa':8} | {'Precision':9} | {'Recall':8} | {'F1/Dice':8}")
print("-" * 90)
for category_name, scores in validation_summary_log.items():
    print(f"{category_name:18} | {scores[0]:.4f}   | {scores[1]:.4f}   | {scores[2]:.4f}  | {scores[3]:.4f}    | {scores[4]:.4f} | {scores[5]:.4f}")
print("==========================================================================================")

Success: Pre-trained weights loaded from path: /content/drive/MyDrive/GLOFEAGLES CHALLENGE/runs/train/fine_tuned/weights/best_finetuned.pt
Detected Target Sub-categories: ['Debris Cover', 'Moraine Dammed', 'Terrain Shadow', 'Varying Turbidity', 'Snow Cover', 'Cloud Cover']

Evaluating Category: Debris Cover
Category metrics calculation completed for: Debris Cover

Evaluating Category: Moraine Dammed
Category metrics calculation completed for: Moraine Dammed

Evaluating Category: Terrain Shadow
Category metrics calculation completed for: Terrain Shadow

Evaluating Category: Varying Turbidity
Category metrics calculation completed for: Varying Turbidity

Evaluating Category: Snow Cover
Category metrics calculation completed for: Snow Cover

Evaluating Category: Cloud Cover
Category metrics calculation completed for: Cloud Cover

                       FINAL COMPREHENSIVE 6-METRIC EVALUATION MATRIX                     
Category Profile   | Accuracy | Mean IoU | Kappa    | Precision | Reca

In [ ]:
import os
import cv2
import shutil
import numpy as np
from sklearn.metrics import jaccard_score

# --- 1. PATH CONFIGURATIONS ---
PRED_ROOT = "/content/drive/MyDrive/GLOFEAGLES CHALLENGE/Validation_Predictions"
DATASET_ROOT = "/content/drive/MyDrive/GLOFEAGLES CHALLENGE/GLOFEagles Challenge Validation Dataset"

IMAGE_ROOT = os.path.join(DATASET_ROOT, "images")
LABEL_ROOT = os.path.join(DATASET_ROOT, "labels")
ANALYSIS_OUTPUT_ROOT = "/content/Top10_Worst_Analysis"

# Verify baseline paths exist
if not os.path.exists(PRED_ROOT) or not os.path.exists(LABEL_ROOT):
    raise FileNotFoundError("Verify your PRED_ROOT or dataset directories are mounted and named correctly.")

# Dynamically locate the categories
categories = [f for f in os.listdir(PRED_ROOT) if os.path.isdir(os.path.join(PRED_ROOT, f))]
print(f"Analyzing performance drops for categories: {categories}\n")

worst_images_by_category = {}

# --- 2. COMPUTE METRICS AND RANK PERFORMANCES ---
for cat in categories:
    cat_pred_dir = os.path.join(PRED_ROOT, cat)
    cat_lbl_dir = os.path.join(LABEL_ROOT, cat)

    pred_files = [f for f in os.listdir(cat_pred_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    image_scores = []

    for filename in pred_files:
        pred_path = os.path.join(cat_pred_dir, filename)
        lbl_path = os.path.join(cat_lbl_dir, filename)

        if os.path.exists(lbl_path):
            pred_mask = cv2.imread(pred_path, cv2.IMREAD_GRAYSCALE)
            gt_mask = cv2.imread(lbl_path, cv2.IMREAD_GRAYSCALE)

            if pred_mask is None or gt_mask is None:
                continue

            # Geometry check to maintain matrix alignment integrity
            if pred_mask.shape != gt_mask.shape:
                gt_mask = cv2.resize(gt_mask, (pred_mask.shape[1], pred_mask.shape[0]), interpolation=cv2.INTER_NEAREST)

            # Flatten to 1D binary vectors
            pred_flat = (pred_mask > 127).astype(np.uint8).flatten()
            gt_flat = (gt_mask > 127).astype(np.uint8).flatten()

            # Calculate intersection over union (IoU)
            iou = jaccard_score(gt_flat, pred_flat, zero_division=1.0)
            image_scores.append({'filename': filename, 'iou': iou})

    # Sort in ascending order (lowest IoU = worst prediction failures)
    image_scores.sort(key=lambda x: x['iou'])
    worst_images_by_category[cat] = image_scores[:10]

# --- 3. EXPORT CORRESPONDING FAILURE ASSETS TO SEPARATE FOLDER ---
print("Isolating and copying worst-performing files to separate folders...")
shutil.rmtree(ANALYSIS_OUTPUT_ROOT, ignore_errors=True) # Reset output workspace

for cat, worst_list in worst_images_by_category.items():
    cat_analysis_dir = os.path.join(ANALYSIS_OUTPUT_ROOT, cat)

    for rank, img_data in enumerate(worst_list, start=1):
        filename = img_data['filename']
        iou_val = img_data['iou']
        stem = os.path.splitext(filename)[0]

        # Create a specific sub-folder for this failure case instance
        case_folder = os.path.join(cat_analysis_dir, f"rank_{rank:02d}_iou_{iou_val:.4f}_{stem}")
        os.makedirs(case_folder, exist_ok=True)

        # Define historical source pathways
        src_img = os.path.join(IMAGE_ROOT, cat, filename)
        src_pred = os.path.join(PRED_ROOT, cat, filename)
        src_lbl = os.path.join(LABEL_ROOT, cat, filename)

        # Safely copy files over with explicit suffix identities
        if os.path.exists(src_img):
            shutil.copy(src_img, os.path.join(case_folder, f"01_original_image_{filename}"))
        if os.path.exists(src_lbl):
            shutil.copy(src_lbl, os.path.join(case_folder, f"02_ground_truth_label_{filename}"))
        if os.path.exists(src_pred):
            shutil.copy(src_pred, os.path.join(case_folder, f"03_your_model_prediction_{filename}"))

print("\n" + "="*70)
print("             DIAGNOSTIC ISOLATION PIPELINE COMPLETE            ")
print("="*70)
for cat, worst_list in worst_images_by_category.items():
    print(f" Category: {cat:18} | Target worst files isolated: {len(worst_list)}")
print(f"\nAll files are organized at: {ANALYSIS_OUTPUT_ROOT}")
print("======================================================================")

Analyzing performance drops for categories: ['Debris Cover', 'Moraine Dammed', 'Terrain Shadow', 'Varying Turbidity', 'Snow Cover', 'Cloud Cover']



In [ ]:
import os
import cv2
import shutil
import numpy as np
from sklearn.metrics import jaccard_score
from tqdm import tqdm  # Visual progress tracker library

# --- 1. PATH CONFIGURATIONS ---
PRED_ROOT = "/content/drive/MyDrive/GLOFEAGLES CHALLENGE/Validation_Predictions"
DATASET_ROOT = "/content/drive/MyDrive/GLOFEAGLES CHALLENGE/GLOFEagles Challenge Validation Dataset"

IMAGE_ROOT = os.path.join(DATASET_ROOT, "images")
LABEL_ROOT = os.path.join(DATASET_ROOT, "labels")
ANALYSIS_OUTPUT_ROOT = "/content/drive/MyDrive/GLOFEAGLES CHALLENGE/Top10_Worst_Analysis"

# Verify baseline paths exist
if not os.path.exists(PRED_ROOT) or not os.path.exists(LABEL_ROOT):
    raise FileNotFoundError("Verify your PRED_ROOT or dataset directories are mounted and named correctly.")

# Dynamically locate the categories
categories = [f for f in os.listdir(PRED_ROOT) if os.path.isdir(os.path.join(PRED_ROOT, f))]
print(f"Discovered Categories for Evaluation: {categories}\n")

worst_images_by_category = {}

# --- 2. COMPUTE METRICS WITH VISUAL PROGRESS TRACKING ---
print("🚀 Commencing matrix intersection scans across categories...")
for cat in categories:
    cat_pred_dir = os.path.join(PRED_ROOT, cat)
    cat_lbl_dir = os.path.join(LABEL_ROOT, cat)

    pred_files = [f for f in os.listdir(cat_pred_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    image_scores = []

    # Wrapped loop with tqdm to visualize the processing speed of the 2,200 images
    for filename in tqdm(pred_files, desc=f"📊 Analyzing {cat:15}"):
        pred_path = os.path.join(cat_pred_dir, filename)
        lbl_path = os.path.join(cat_lbl_dir, filename)

        if os.path.exists(lbl_path):
            pred_mask = cv2.imread(pred_path, cv2.IMREAD_GRAYSCALE)
            gt_mask = cv2.imread(lbl_path, cv2.IMREAD_GRAYSCALE)

            if pred_mask is None or gt_mask is None:
                continue

            # Geometry check to maintain matrix alignment integrity
            if pred_mask.shape != gt_mask.shape:
                gt_mask = cv2.resize(gt_mask, (pred_mask.shape[1], pred_mask.shape[0]), interpolation=cv2.INTER_NEAREST)

            # Flatten to 1D binary vectors
            pred_flat = (pred_mask > 127).astype(np.uint8).flatten()
            gt_flat = (gt_mask > 127).astype(np.uint8).flatten()

            # Calculate intersection over union (IoU)
            iou = jaccard_score(gt_flat, pred_flat, zero_division=1.0)
            image_scores.append({'filename': filename, 'iou': iou})

    # Sort in ascending order (lowest IoU = worst prediction failures)
    image_scores.sort(key=lambda x: x['iou'])
    worst_images_by_category[cat] = image_scores[:10]

# --- 3. EXPORT CORRESPONDING FAILURE ASSETS TO SEPARATE FOLDER ---
print("\n Isolating and copying target worst-performing files to storage...")
shutil.rmtree(ANALYSIS_OUTPUT_ROOT, ignore_errors=True) # Reset output workspace

# Tracking the copy loop for the 60 target files (10 per category)
for cat, worst_list in tqdm(worst_images_by_category.items(), desc="Exporting Assets "):
    cat_analysis_dir = os.path.join(ANALYSIS_OUTPUT_ROOT, cat)

    for rank, img_data in enumerate(worst_list, start=1):
        filename = img_data['filename']
        iou_val = img_data['iou']
        stem = os.path.splitext(filename)[0]

        # Create a specific sub-folder for this failure case instance
        case_folder = os.path.join(cat_analysis_dir, f"rank_{rank:02d}_iou_{iou_val:.4f}_{stem}")
        os.makedirs(case_folder, exist_ok=True)

        # Define historical source pathways
        src_img = os.path.join(IMAGE_ROOT, cat, filename)
        src_pred = os.path.join(PRED_ROOT, cat, filename)
        src_lbl = os.path.join(LABEL_ROOT, cat, filename)

        # Safely copy files over with explicit suffix identities
        if os.path.exists(src_img):
            shutil.copy(src_img, os.path.join(case_folder, f"01_original_image_{filename}"))
        if os.path.exists(src_lbl):
            shutil.copy(src_lbl, os.path.join(case_folder, f"02_ground_truth_label_{filename}"))
        if os.path.exists(src_pred):
            shutil.copy(src_pred, os.path.join(case_folder, f"03_your_model_prediction_{filename}"))

print("\n" + "="*70)
print("             DIAGNOSTIC ISOLATION PIPELINE COMPLETE            ")
print("="*70)
for cat, worst_list in worst_images_by_category.items():
    print(f" Category: {cat:18} | Target worst files isolated: {len(worst_list)}")
print(f"\nAll files are organized at: {ANALYSIS_OUTPUT_ROOT}")
print("======================================================================")

Discovered Categories for Evaluation: ['Debris Cover', 'Moraine Dammed', 'Terrain Shadow', 'Varying Turbidity', 'Snow Cover', 'Cloud Cover']

🚀 Commencing matrix intersection scans across categories...


📊 Analyzing Cloud Cover    : 100%|██████████| 12/12 [00:08<00:00,  1.41it/s]



📦 Isolating and copying target worst-performing files to storage...


Exporting Assets : 100%|██████████| 6/6 [00:31<00:00,  5.28s/it]


             DIAGNOSTIC ISOLATION PIPELINE COMPLETE            
 Category: Debris Cover       | Target worst files isolated: 10
 Category: Moraine Dammed     | Target worst files isolated: 10
 Category: Terrain Shadow     | Target worst files isolated: 10
 Category: Varying Turbidity  | Target worst files isolated: 10
 Category: Snow Cover         | Target worst files isolated: 10
 Category: Cloud Cover        | Target worst files isolated: 10

All files are organized at: /content/drive/MyDrive/GLOFEAGLES CHALLENGE/Top10_Worst_Analysis
